In [ ]:
# import os

# # Récupérer le répertoire courant
# repertoire_courant = os.getcwd()

# # Liste tous les fichiers dans le répertoire courant
# fichiers = [fichier for fichier in os.listdir(repertoire_courant) if fichier.endswith('.csv')]

# # Supprime chaque fichier
# for fichier in fichiers:
#     chemin_fichier = os.path.join(repertoire_courant, fichier)
#     os.remove(chemin_fichier)

# print("Tous les fichiers CSV ont été supprimés du répertoire courant.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Normalise les données, recherche le meilleur paramètre epsilon pour DBSCAN en utilisant la silhouette score, puis applique DBSCAN pour regrouper les séries temporelles.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Charger le fichier CSV
data = pd.read_csv("/content/drive/MyDrive/TestsGooC/donnees_fusionnees14D.csv")
# data = pd.read_csv("/content/-1.csv")
print(len(data))
# Supprimer la colonne contenant les noms des fonctions
functions = data.pop('HashFunction')

# Normalisation des données
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

# Détermination du meilleur paramètre epsilon pour DBSCAN
eps_values = np.linspace(0.1, 80, 100)
silhouette_scores = []

for eps in eps_values:
    print(eps)
    dbscan = DBSCAN(eps=eps, min_samples=2)
    dbscan.fit(scaled_data)
    labels = dbscan.labels_
    if len(np.unique(labels)) > 1:  # Ignorer les valeurs qui ne donnent qu'un seul cluster
        silhouette_scores.append(silhouette_score(scaled_data, labels))
    else:
        silhouette_scores.append(-1)  # Assigner une silhouette score de -1 si un seul cluster est trouvé

best_eps = eps_values[np.argmax(silhouette_scores)]
print("Meilleur epsilon trouvé:", best_eps)

# Clustering avec DBSCAN en utilisant le meilleur epsilon trouvé
dbscan = DBSCAN(eps=best_eps, min_samples=2)
# dbscan = DBSCAN(eps=2.0, min_samples=2)
clusters = dbscan.fit_predict(scaled_data)

# Afficher les résultats du clustering
clustered_data = pd.DataFrame({'function': functions, 'cluster': clusters})
print(clustered_data)

1309
0.1
0.9070707070707071
1.7141414141414144
2.5212121212121215
3.3282828282828287
4.135353535353535
4.942424242424242
5.74949494949495
6.556565656565657
7.363636363636364
8.17070707070707
8.977777777777778
9.784848484848485
10.591919191919192
11.3989898989899
12.206060606060607
13.013131313131314
13.820202020202021
14.627272727272729
15.434343434343436
16.241414141414143
17.04848484848485
17.855555555555558
18.662626262626265
19.469696969696972
20.27676767676768
21.083838383838387
21.890909090909094
22.6979797979798
23.50505050505051
24.312121212121216
25.119191919191923
25.92626262626263
26.733333333333338
27.540404040404045
28.347474747474752
29.15454545454546
29.961616161616167
30.768686868686874
31.57575757575758
32.382828282828285
33.18989898989899
33.9969696969697
34.80404040404041
35.611111111111114
36.41818181818182
37.22525252525253
38.032323232323236
38.83939393939394
39.64646464646465
40.45353535353536
41.260606060606065
42.06767676767677
42.87474747474748
43.681818181818

In [ ]:
# Calculer le nombre de fonctions dans chaque cluster
unique_clusters, counts = np.unique(clusters, return_counts=True)

# Afficher le nombre de fonctions dans chaque cluster
for cluster_id, count in zip(unique_clusters, counts):
    print(f"Cluster {cluster_id}: {count} fonctions")

Cluster -1: 239 fonctions
Cluster 0: 1027 fonctions
Cluster 1: 3 fonctions
Cluster 2: 6 fonctions
Cluster 3: 2 fonctions
Cluster 4: 3 fonctions
Cluster 5: 2 fonctions
Cluster 6: 3 fonctions
Cluster 7: 2 fonctions
Cluster 8: 2 fonctions
Cluster 9: 2 fonctions
Cluster 10: 3 fonctions
Cluster 11: 2 fonctions
Cluster 12: 4 fonctions
Cluster 13: 2 fonctions
Cluster 14: 3 fonctions
Cluster 15: 2 fonctions
Cluster 16: 2 fonctions


In [ ]:
# Enregistrer chaque cluster dans un fichier CSV distinct
for cluster_id in np.unique(clusters):
    cluster_indices = np.where(clusters == cluster_id)
    cluster_data = data.iloc[cluster_indices[0]]
    cluster_functions = functions.iloc[cluster_indices[0]]
    cluster_df = pd.concat([cluster_functions, cluster_data], axis=1)
    cluster_df.to_csv(f"cluster_{cluster_id}.csv", index=False)
    print(f"Cluster {cluster_id} enregistré dans cluster_{cluster_id}.csv")

Cluster -1 enregistré dans cluster_-1.csv
Cluster 0 enregistré dans cluster_0.csv
Cluster 1 enregistré dans cluster_1.csv
Cluster 2 enregistré dans cluster_2.csv
Cluster 3 enregistré dans cluster_3.csv
Cluster 4 enregistré dans cluster_4.csv
Cluster 5 enregistré dans cluster_5.csv
Cluster 6 enregistré dans cluster_6.csv
Cluster 7 enregistré dans cluster_7.csv
Cluster 8 enregistré dans cluster_8.csv
Cluster 9 enregistré dans cluster_9.csv
Cluster 10 enregistré dans cluster_10.csv
Cluster 11 enregistré dans cluster_11.csv
Cluster 12 enregistré dans cluster_12.csv
Cluster 13 enregistré dans cluster_13.csv
Cluster 14 enregistré dans cluster_14.csv
Cluster 15 enregistré dans cluster_15.csv
Cluster 16 enregistré dans cluster_16.csv


In [ ]:
# # Visualisation des clusters
# plt.figure(figsize=(10, 6))
# for cluster_id in np.unique(clusters):
#     cluster_indices = np.where(clusters == cluster_id)
#     for i in cluster_indices[0]:
#         plt.plot(data.iloc[i])
#     plt.title(f'Cluster {cluster_id}')
#     plt.xlabel('Minute')
#     plt.ylabel('Nombre d\'appels')
#     plt.show()

In [ ]:
print(np.unique(clusters))

[-1  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]


In [ ]:
# # Visualisation des clusters pour les clusters dans la plage (2, 10)
# plt.figure(figsize=(10, 6))
# for cluster_id in range(0, 1):
#     cluster_indices = np.where(clusters == cluster_id)
#     for i in cluster_indices[0]:
#         plt.plot(data.iloc[i])
#     plt.title(f'Cluster {cluster_id}')
#     plt.xlabel('Minute')
#     plt.ylabel('Nombre d\'appels')
#     plt.show()

In [ ]:
# Analyse statistique globale de chaque cluster
for cluster_id in unique_clusters:
    cluster_indices = np.where(clusters == cluster_id)
    cluster_data = data.iloc[cluster_indices[0]]

    print(f"\nCluster {cluster_id} - Nombre de fonctions : {len(cluster_indices[0])}")

    # Calcul des statistiques globales pour toutes les séries temporelles dans ce cluster
    cluster_stats = describe(cluster_data.to_numpy().flatten())

    print("\nAnalyse statistique globale des séries temporelles dans le cluster :")
    print("Nombre d'observations :", cluster_stats.nobs)
    print("Moyenne :", cluster_stats.mean)
    print("Écart-type :", cluster_stats.variance**0.5)
    print("Minimum :", cluster_stats.minmax[0])
    print("Maximum :", cluster_stats.minmax[1])
    print("Skewness :", cluster_stats.skewness)
    print("Kurtosis :", cluster_stats.kurtosis)


Cluster -1 - Nombre de fonctions : 239


NameError: name 'describe' is not defined

Copie de fichiers

In [ ]:
import shutil
import os

def copier_fichier(chemin_source, chemin_destination):
    try:
        shutil.copy(chemin_source, chemin_destination)
        print("Le fichier a été copié avec succès.")
    except IOError as e:
        print(f"Erreur lors de la copie du fichier : {e}")
    except Exception as e:
        print(f"Une erreur inattendue s'est produite : {e}")

# Exemple d'utilisation
fichier_source = "/content/cluster_0.csv"
repertoire_destination = "/content/drive/MyDrive/TestsGooC/tests 08 04"

if os.path.isfile(fichier_source):
  copier_fichier(fichier_source, repertoire_destination)
else:
  print("Le fichier source n'existe pas.")

Le fichier a été copié avec succès.


In [ ]:
# import shutil
# import os

# def copier_fichiers(chemin_source, chemin_destination):
#     try:
#         for i in range(-1, 265):
#             fichier_source = f"{chemin_source}/cluster_{i}.csv"
#             if os.path.isfile(fichier_source):
#                 shutil.copy(fichier_source, chemin_destination)
#                 print(f"Le fichier {fichier_source} a été copié avec succès.")
#             else:
#                 print(f"Le fichier {fichier_source} n'existe pas.")
#     except Exception as e:
#         print(f"Une erreur inattendue s'est produite : {e}")

# # Exemple d'utilisation
# repertoire_source = "/content"  # Chemin du répertoire contenant les fichiers cluster_-1 à cluster_100
# repertoire_destination = "/content/drive/MyDrive/TestsGooC/tests 08 04/ClustersDBSCAN/C0"

# copier_fichiers(repertoire_source, repertoire_destination)
